# Establishing a pipeline for semantic matching

Will be implemented into a larger pipeline that factors in the location and network of the individuals

In [1]:
import torch
from sentence_transformers import SentenceTransformer, util

def generate_embeddings(text_list: list[str], model: SentenceTransformer) -> torch.Tensor:
    """
    Generates and returns cached dense tensors for a pool of strings.
    Abstracts the implmentation of the model, in case the model output is not exactly as expected
    """
    return model.encode(text_list, convert_to_tensor=True)



e:\dev\workspace\Apziva\Project3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def score_off_keywords(keyword_embeddings: torch.Tensor, candidate_embeddings: torch.Tensor,) -> list[float]:
    """
    Ranks candidates using pre-calculated tensors against an array of search keywords.
    Returns a unified list of match scores (0.0 to 1.0) for every candidate.
    """
    # Average the search keywords if there are multiple inputs
    if keyword_embeddings.ndim > 1:
        base_search_vector = torch.mean(keyword_embeddings, dim=0)
    else:
        base_search_vector = keyword_embeddings

    # Compute cosine similarity against all candidates in a single matrix operation
    # util.cos_sim returns a matrix shape [1, num_candidates]; we flatten it using [0]
    scores = util.cos_sim(base_search_vector, candidate_embeddings)[0]
    
    return scores.tolist()


In [3]:
def rescore_candidates(good_idxs: list[int], candidate_embeddings: torch.Tensor, keyword_embeddings: torch.Tensor, alpha: float = 0.4) -> list[float]:
    """
    params:
        good_ids: list of promising candidate's indexes in candidate_embeddings tensor
        candidate_embeddings: tensor of candidates
        keyword_embeddings: tensor of keywords
        alpha: weighting (float in range 0-1) of how much to change promising candidate search vector based on promising candidates' ids

    Blends keyword intent with an average vector of successful candidate profiles.
    Returns a newly adjusted list of match scores for the dataset pool.
    """
    # generate the base search vector baseline
    if keyword_embeddings.ndim > 1:
        base_search_vector = torch.mean(keyword_embeddings, dim=0)
    else:
        base_search_vector = keyword_embeddings

    # If no promising candidates have been flagged yet, return baseline scores
    if not good_idxs:
        scores = util.cos_sim(base_search_vector, candidate_embeddings)[0]
        return scores.tolist()

    # Pull the exact precalculated tensors for selected candidates and average them
    selected_vectors = candidate_embeddings[good_idxs]
    avg_promising_vector = torch.mean(selected_vectors, dim=0)

    # Synthesize the updated search vector profile
    updated_search_vector = ((1 - alpha) * base_search_vector) + (alpha * avg_promising_vector)

    # Calculate the adjusted search metrics across the pool
    reranked_scores = util.cos_sim(updated_search_vector, candidate_embeddings)[0]
    
    return reranked_scores.tolist()


In [4]:
import pandas as pd

# get data
df = pd.read_csv("../../data/processed/potential-talents-no-fit-col.csv")

text_df = df[["id","job_title"]]

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

# Workflow
keywords = ["aspiring human resources", "HR"]
model = SentenceTransformer('TechWolf/JobBERT-v2') # fine tuned for job listings

# get embeddings
keyword_embeddings = generate_embeddings(text_list=keywords,model=model)
candidate_embeddings = generate_embeddings(text_list=df["job_title"].tolist(), model=model)


df["job_title_score"] = score_off_keywords(keyword_embeddings=keyword_embeddings, candidate_embeddings=candidate_embeddings)

# they really liked candidates with ids 4 and 7
good_ids = [4,7]
good_idxs = [df[df["id"] == id].index for id in good_ids]

df["job_title_rescored"] = rescore_candidates(good_idxs=good_idxs, candidate_embeddings=candidate_embeddings, keyword_embeddings=keyword_embeddings, alpha=0.4)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1751.33it/s]
C:\Users\16163\AppData\Local\Temp\ipykernel_39544\2005537558.py:24: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:362.)
  selected_vectors = candidate_embeddings[good_idxs]


In [9]:
from pprint import pprint
pprint([df[df["id"] == id]["job_title"] for id in good_ids])

[3    People Development Coordinator at Ryan
Name: job_title, dtype: str,
 6    Student at Humber College and Aspiring Human R...
Name: job_title, dtype: str]


In [10]:
df

,id,job_title,location,connection,job_title_score,job_title_rescored
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,0.704916,0.700161
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,0.234922,0.229963
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.817956,0.806440
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,0.631290,0.624242
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,0.231718,0.233102
...,...,...,...,...,...,...
99,100,Aspiring Human Resources Manager | Graduating ...,"Cape Girardeau, Missouri",103,0.748668,0.742287
100,101,Human Resources Generalist at Loparex,"Raleigh-Durham, North Carolina Area",500+,0.819746,0.812826
101,102,Business Intelligence and Analytics at Travelers,Greater New York City Area,49,0.305845,0.307591
102,103,Always set them up for Success,Greater Los Angeles Area,500+,0.328669,0.323421


In [12]:
df.sort_values("job_title_score", ascending=False)

,id,job_title,location,connection,job_title_score,job_title_rescored
5,6,Aspiring Human Resources Specialist,Greater New York City Area,1,0.830705,0.818875
23,24,Aspiring Human Resources Specialist,Greater New York City Area,1,0.830705,0.818875
35,36,Aspiring Human Resources Specialist,Greater New York City Area,1,0.830705,0.818875
48,49,Aspiring Human Resources Specialist,Greater New York City Area,1,0.830705,0.818875
59,60,Aspiring Human Resources Specialist,Greater New York City Area,1,0.830705,0.818875
...,...,...,...,...,...,...
41,42,"SVP, CHRO, Marketing & Communications, CSR Off...","Houston, Texas Area",500+,0.200772,0.196875
63,64,"SVP, CHRO, Marketing & Communications, CSR Off...","Houston, Texas Area",500+,0.200772,0.196875
54,55,"SVP, CHRO, Marketing & Communications, CSR Off...","Houston, Texas Area",500+,0.200772,0.196875
92,93,Admissions Representative at Community medical...,"Long Beach, California",9,0.175789,0.170790


In [13]:
df.sort_values("job_title_rescored", ascending=False)

,id,job_title,location,connection,job_title_score,job_title_rescored
5,6,Aspiring Human Resources Specialist,Greater New York City Area,1,0.830705,0.818875
23,24,Aspiring Human Resources Specialist,Greater New York City Area,1,0.830705,0.818875
35,36,Aspiring Human Resources Specialist,Greater New York City Area,1,0.830705,0.818875
48,49,Aspiring Human Resources Specialist,Greater New York City Area,1,0.830705,0.818875
59,60,Aspiring Human Resources Specialist,Greater New York City Area,1,0.830705,0.818875
...,...,...,...,...,...,...
41,42,"SVP, CHRO, Marketing & Communications, CSR Off...","Houston, Texas Area",500+,0.200772,0.196875
63,64,"SVP, CHRO, Marketing & Communications, CSR Off...","Houston, Texas Area",500+,0.200772,0.196875
54,55,"SVP, CHRO, Marketing & Communications, CSR Off...","Houston, Texas Area",500+,0.200772,0.196875
92,93,Admissions Representative at Community medical...,"Long Beach, California",9,0.175789,0.170790
